In [1]:
import os
import pandas as pd
from maomao.parsing.parsing_utils import *
from maomao.utils.constants import *

import warnings
warnings.filterwarnings('ignore')

#### Processing and standardizing peptide datasets (DRAMP)

This notebook curates two toxicity-related datasets from **DRAMP** using the `general_amps.xlsx` export. DRAMP provides free-text annotations for hemolytic activity and cytotoxicity; here we apply a rule-based labeling strategy to convert those text fields into binary labels, then perform duplicate consistency checks and export standardized datasets and metadata.

- **Toxic effect / endpoint:** hemolytic, cytolytic and cytotoxic
- **Source:** DRAMP
- **Sequence scope:** only non-modified peptide sequences are retained for the final dataset.

The pipeline performs the following steps:

- **Loads the DRAMP Excel export** (`general_amps.xlsx`).
- **Derives a hemolysis label (`label`) from the `Hemolytic_activity` column**:
  - assigns `1` when the text indicates hemolytic activity (e.g., contains “hemolytic”),
  - assigns `0` when the text explicitly indicates absence of hemolysis (e.g., “no hemolytic”, “non-hemolytic”, “no detectable/significant hemolytic”),
  - leaves entries as `NA` when the annotation is missing or ambiguous (these are dropped).
- **Derives a cytotoxicity label (`label`) from the `Cytotoxicity` column**:
  - assigns `0` for explicit non-cytotoxic statements (e.g., “no cytotoxicity”, “similar to control”, “no significant difference”),
  - assigns `1` when cytotoxicity is reported (contains “cytotoxicity”) and is not a “no information found” case,
  - drops entries with `NA`.
- **Builds two curated tables** with the unified schema `(sequence, label)`:
  - `processed_hemolytic_dataset.csv`
  - `processed_cytotoxic_dataset.csv`
- **Checks duplicates by sequence** for each subset:
  - unique sequences are kept,
  - duplicates with consistent labels are collapsed,
  - sequences with conflicting labels are flagged and exported as errors.
- **Builds metadata** from the project-wide Excel description sheet and appends QC statistics.
- **Exports** the curated datasets, an error report, and `metadata.json`.

In [2]:
name_source = "DRAMP"
name_task = "toxic_effect_classification"

# PATH_INPUT and PATH_EXPORT are imported from maomao.utils.constants
# Update them in constants.py according to the required input and export paths.

- Reading raw data

In [3]:
df = pd.read_excel(f"{PATH_INPUT}/{name_source}/general_amps.xlsx")

In [4]:
df_hemolytic = (
    df.assign(hemolytic_label=pd.NA)
)

# hemolytic → 1
df_hemolytic.loc[
    df_hemolytic["Hemolytic_activity"].str.contains("hemolytic", case=False, na=False),
    "hemolytic_label"
] = 1

# no / non / not / no detectable / no significant hemolytic → 0
df_hemolytic.loc[
    df_hemolytic["Hemolytic_activity"].str.contains(
        r"(no\s*hemolytic"
        r"|non[-\s]*hemolytic"
        r"|not\s*hemolytic"
        r"|no\s*detectable\s*hemolytic"
        r"|no\s*significant\s*hemolytic)",
        case=False,
        na=False
    ),
    "hemolytic_label"
] = 0

In [5]:
df_hemolytic = (
    df_hemolytic
    .rename(columns={"Sequence": "sequence", "hemolytic_label": "label"})
    [["sequence", "label"]]
)
df_hemolytic = df_hemolytic[~df_hemolytic["label"].isna()]
df_hemolytic.shape

(511, 2)

In [6]:
df_cytotoxic = df.assign(cytotoxicity_label=pd.NA)  # Inicializa la columna con NA

# Non cytotoxic → 0
df_cytotoxic.loc[
    df_cytotoxic["Cytotoxicity"].str.contains(
        r"(no\s*cytotoxicity"
        r"|non[-\s]*cytotoxicity"
        r"|not\s*cytotoxicity"
        r"|no\s*detectable\s*cytotoxicity"
        r"|no\s*significant\s*cytotoxicity"
        r"|not\s*significantly\s*different"
        r"|no\s*significant\s*difference"
        r"|similar\s*to\s*control)", 
        case=False, 
        na=False
    ),
    "cytotoxicity_label"
] = 0

# Cytotoxic → 1
df_cytotoxic.loc[
    df_cytotoxic["Cytotoxicity"].str.contains(
        r"\bcytotoxicity\b", case=False, na=False
    )
    & ~df_cytotoxic["Cytotoxicity"].str.contains(
        r"(no\s*cytotoxicity\s*information\s*found)", 
        case=False, 
        na=False
    )
    & df_cytotoxic["cytotoxicity_label"].isna(),
    "cytotoxicity_label"
] = 1

In [7]:
df_cytotoxic = (
    df_cytotoxic
    .rename(columns={"Sequence": "sequence", "cytotoxicity_label": "label"})
    [["sequence", "label"]]
)
df_cytotoxic = df_cytotoxic[~df_cytotoxic["label"].isna()]
df_cytotoxic.shape

(2776, 2)

In [8]:
df_cytolytic = df.assign(cytolytic_label=pd.NA)

# Non cytolytic → 0
df_cytolytic.loc[
    df_cytolytic["Comments"].str.contains(
        r"(no\s+cytolytic"
        r"|no\s+hemolytic\s+nor\s+cytolytic"
        r"|no\s+detectable\s+cytolytic"
        r"|lacks\s+cytolytic)",
        case=False,
        na=False
    ),
    "cytolytic_label"
] = 0
# Cytolytic → 1
df_cytolytic.loc[
    df_cytolytic["Comments"].str.contains(
        r"(cytolytic\s+activity"
        r"|cytolysis"
        r"|membrane\s+lysis"
        r"|cell\s+membrane\s+lysis"
        r"|forms\s+pores"
        r"|pore[-\s]*forming"
        r"|permeabilizes\s+membrane"
        r"|ion[-\s]*permeable\s+channels)",
        case=False,
        na=False
    )
    & df_cytolytic["cytolytic_label"].isna(),
    "cytolytic_label"
] = 1

In [9]:
df_cytolytic = (
    df_cytolytic
    .rename(columns={"Sequence": "sequence", "cytolytic_label": "label"})
    [["sequence", "label"]]
)
df_cytolytic = df_cytolytic[~df_cytolytic["label"].isna()]
df_cytolytic.shape

(80, 2)

- Checking duplicates

In [10]:
df_remove_duplicated_cytotoxic, df_errors_cytotoxic, df_unique_cytotoxic = processing_duplicated(df_cytotoxic, group_seq="sequence", sort_key="label")

In [11]:
df_remove_duplicated_hemolytic, df_errors_hemolytic, df_unique_hemolytic = processing_duplicated(df_hemolytic, group_seq="sequence", sort_key="label")

In [12]:
df_remove_duplicated_cytolytic, df_errors_cytolytic, df_unique_cytolytic = processing_duplicated(df_cytolytic, group_seq="sequence", sort_key="label")

In [13]:
df_full_cytotoxic = pd.concat([df_unique_cytotoxic, df_remove_duplicated_cytotoxic])
df_full_hemolytic = pd.concat([df_unique_hemolytic, df_remove_duplicated_hemolytic])
df_full_cytolytic = pd.concat([df_unique_cytolytic, df_remove_duplicated_cytolytic])

df_full = pd.concat([df_full_cytotoxic, df_full_hemolytic, df_full_cytolytic])
df_errors = pd.concat([df_errors_cytotoxic, df_errors_hemolytic, df_errors_cytolytic])

- Working with metada

In [14]:
df_metada = read_metadata("../../raw_data/raw_data_description.xlsx", name_source)
dict_metadata = create_metada_with_multiple_values(df_metada)

In [15]:
dict_metadata.update({
    "number_of_raw_sequences": int(len(df)),
    "number_of_sequences_retained": len(df_full),
    "number_of_positive_sequences": int((df_full["label"] == 1).sum()),
    "number_of_negative_sequences": int((df_full["label"] == 0).sum()),
    "number_of_erroneous_sequences": int(len(df_errors)),
    "modified_sequences_included": False,
})

dict_metadata

{'type source': 'Database',
 'static-dynamic': 'Dynamic',
 'license': 'Creative Commons Attribution 4.0',
 'year of publication': 2025,
 'last update date': datetime.datetime(2025, 7, 16, 0, 0),
 'download date': Timestamp('2025-08-01 00:00:00'),
 'file format': 'xlsx',
 'peptide property': 'anticancer, antiviral, antiplasmodial, cytotoxic, antibacterial, antimicrobial, antifungal, antiparasitic, insecticidal, wound healing, enzyme inhibitor, anti mammalian cells, antibiotic, antidiabetic, antimalaria, anti HIV, targeting mammals, antiinflammatory, toxic, antiproliferative, anticandidal, antiprotozoal, hemolytic, anti SaRS-CoV 2, anti coronaviridae, chemotactic, proteolytic, defense response, protein kinase inhibitor activity, cell degranulating, non-antibacterial, non-antimicrobial, anticryptococcal, archaeolytic, antilisterial, immunomodulating, cytolytic, antiallodynic, anti gram negative, anti gram positive',
 'dataset information': 'Positive',
 'unit of measurement': 'No informati

- Exporting data

In [16]:
os.makedirs(f"{PATH_EXPORT}/{name_task}/{name_source}/", exist_ok=True)
export_json(f"{PATH_EXPORT}/{name_task}/{name_source}/metadata.json", dict_metadata)

In [17]:
df_full_hemolytic.to_csv(f"{PATH_EXPORT}/{name_task}/{name_source}/processed_hemolytic_dataset.csv", index=False)
df_full_cytotoxic.to_csv(f"{PATH_EXPORT}/{name_task}/{name_source}/processed_cytotoxic_dataset.csv", index=False)
df_full_cytolytic.to_csv(f"{PATH_EXPORT}/{name_task}/{name_source}/processed_cytolytic_dataset.csv", index=False)


df_errors.to_csv(f"{PATH_EXPORT}/{name_task}/{name_source}/detected_error_sequences.csv", index=False)